# 03 — Model Training

We fine-tune `xx_ent_wiki_sm` on the 186 labeled resumes.

The `tok2vec` layer stays frozen — only the NER layer is trained.  
We save the model whenever F1 improves on the dev set.

In [ ]:
import json
import random
from pathlib import Path

import spacy
from spacy.training import Example
from spacy.util import minibatch, compounding
import matplotlib.pyplot as plt

colors = ['#4361ee', '#3a86ff', '#ff006e', '#fb5607']

# load the processed data from notebook 01
train_data = json.loads(Path('../data/processed/train_data.json').read_text(encoding='utf-8'))
test_data  = json.loads(Path('../data/processed/test_data.json').read_text(encoding='utf-8'))

print(f'Train : {len(train_data)} resumes')
print(f'Test  : {len(test_data)} resumes')

## Hyperparameters

| Parameter | Value | Reason |
|-----------|-------|--------|
| epochs | 30 | Enough for this dataset size; we keep the best checkpoint |
| dropout | 0.35 | Prevents overfitting on the small 186-example dataset |
| batch size | 4 → 32 (compounding) | Starts small for stability, grows gradually |
| learning rate | 0.001 | spaCy's recommended default for NER fine-tuning |
| frozen layers | tok2vec | We keep the Wikipedia-pretrained representations intact |

In [ ]:
N_ITER  = 30
DROPOUT = 0.35

random.seed(42)

print('Loading xx_ent_wiki_sm...')
nlp = spacy.load('xx_ent_wiki_sm')

# add the resume-specific entity labels to the NER component
ner = nlp.get_pipe('ner')
all_labels = set()
for item in train_data:
    for _, _, label in item['entities']:
        all_labels.add(label)

for label in sorted(all_labels):
    ner.add_label(label)

print(f'Labels added ({len(all_labels)}):')
print(sorted(all_labels))

In [ ]:
def prepare_examples(nlp, data):
    """Convert our JSON data into spaCy Example objects for training."""
    examples = []

    for item in data:
        doc  = nlp.make_doc(item['text'])
        ents = []

        for start, end, label in item['entities']:
            # char_span converts character offsets to token-level spans
            # alignment_mode='expand' handles cases where offsets land mid-token
            span = doc.char_span(start, end, label=label, alignment_mode='expand')
            if span is not None:
                ents.append(span)

        # drop any overlaps that may have appeared after the expand alignment
        clean_ents = []
        for span in sorted(ents, key=lambda s: s.start):
            has_overlap = any(
                span.start < prev.end and span.end > prev.start
                for prev in clean_ents
            )
            if not has_overlap:
                clean_ents.append(span)

        ref_doc = doc.copy()
        ref_doc.ents = clean_ents
        examples.append(Example(doc, ref_doc))

    return examples


train_examples = prepare_examples(nlp, train_data)
dev_examples   = prepare_examples(nlp, test_data)

print(f'Train examples : {len(train_examples)}')
print(f'Dev examples   : {len(dev_examples)}')

In [ ]:
model_output_path   = Path('../fine_tuning/model_output')
metrics_output_path = Path('../fine_tuning/training_metrics.json')
model_output_path.parent.mkdir(parents=True, exist_ok=True)

history = []
best_f1 = 0.0

# disable all pipes except NER during training so their weights don't update
other_pipes = [pipe for pipe in nlp.pipe_names if pipe != 'ner']

print('Starting training...')
print('-' * 55)

with nlp.disable_pipes(*other_pipes):
    optimizer = nlp.resume_training()
    optimizer.learn_rate = 0.001

    for epoch in range(1, N_ITER + 1):
        random.shuffle(train_examples)
        losses = {}

        # compounding batch size grows from 4 to 32 gradually
        batches = minibatch(train_examples, size=compounding(4.0, 32.0, 1.001))
        for batch in batches:
            nlp.update(batch, drop=DROPOUT, losses=losses, sgd=optimizer)

        # evaluate on the dev set after each epoch
        scores = nlp.evaluate(dev_examples)
        p    = round(scores['ents_p'] * 100, 2)
        r    = round(scores['ents_r'] * 100, 2)
        f1   = round(scores['ents_f'] * 100, 2)
        loss = round(losses.get('ner', 0), 4)

        history.append({'epoch': epoch, 'loss': loss, 'precision': p, 'recall': r, 'f1': f1})
        print(f'Epoch {epoch:2d}  |  loss={loss:.4f}  |  P={p:.1f}  R={r:.1f}  F1={f1:.1f}')

        # save only when we get a better F1
        if f1 > best_f1:
            best_f1 = f1
            model_output_path.mkdir(exist_ok=True)
            nlp.to_disk(model_output_path)

with open(metrics_output_path, 'w') as f:
    json.dump(history, f, indent=2)

print('-' * 55)
print(f'Training done. Best F1: {best_f1:.2f}')
print(f'Model saved to: {model_output_path}')

In [ ]:
epochs = [h['epoch']     for h in history]
losses = [h['loss']      for h in history]
f1s    = [h['f1']        for h in history]
ps     = [h['precision'] for h in history]
rs     = [h['recall']    for h in history]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Training Progress', fontweight='bold', fontsize=13)

# loss curve
axes[0].plot(epochs, losses, color=colors[2], linewidth=2, marker='o', markersize=3)
axes[0].fill_between(epochs, losses, alpha=0.1, color=colors[2])
axes[0].set_title('NER Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(alpha=0.3)

# precision / recall / F1
axes[1].plot(epochs, ps,  color=colors[0], linewidth=2,   label='Precision', marker='o', markersize=3)
axes[1].plot(epochs, rs,  color=colors[1], linewidth=2,   label='Recall',    marker='s', markersize=3)
axes[1].plot(epochs, f1s, color=colors[3], linewidth=2.5, label='F1',        marker='^', markersize=4)
best_epoch = epochs[f1s.index(max(f1s))]
axes[1].axvline(best_epoch, color='gray', linestyle='--', linewidth=1.2,
                label=f'Best epoch ({best_epoch})')
axes[1].set_title('Precision / Recall / F1 on Dev Set')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Score (%)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()